# Detectron2: seis tareas de visión por computadora y una API

Este notebook demuestra cómo usar **Detectron2** (Meta AI Research) con modelos preentrenados del *Model Zoo* para resolver seis tareas:

| Tarea | Modelo | Qué devuelve |
|---|---|---|
| Detección de objetos | Faster R-CNN R50-FPN | Cajas `[x1, y1, x2, y2]`, clase y confianza |
| Segmentación de instancias | Mask R-CNN R50-FPN | Lo anterior + una máscara binaria por objeto |
| Segmentación semántica | Panoptic FPN R101 (derivada) | Un mapa donde cada píxel tiene una clase |
| Segmentación panóptica | Panoptic FPN R101 | Mapa de segmentos: objetos individuales + fondo |
| Keypoints | Keypoint R-CNN R50-FPN | 17 puntos `(x, y, confianza)` por persona |
| DensePose | DensePose R-CNN R50-FPN | Por cada píxel de la persona: parte del cuerpo y coordenadas `(U, V)` sobre la superficie 3D |

Al final se levanta una **API Flask** que recibe una imagen y el tipo de tarea, y devuelve el resultado en JSON listo para dibujar en una página web.

## Requisitos del pod en RunPod

- Plantilla **RunPod PyTorch 2.x** (trae CUDA, PyTorch y Jupyter).
- GPU con 16 GB o más (RTX A4000, A5000, 4090). La inferencia de los seis modelos cabe holgadamente en 24 GB.
- En la configuración del pod, en **Expose HTTP Ports**, agregar el puerto **8000** (además del 8888 de Jupyter). La API quedará en `https://<POD_ID>-8000.proxy.runpod.net`.
- Disco: unos 10 GB libres en `/workspace` (repositorio, compilación y pesos).

## 1. Instalación

Esta celda hace todo en un paso y muestra el avance:

1. Instala las dependencias (con un arreglo para `blinker`, que en la imagen de RunPod viene instalado por el sistema y pip no puede reemplazar).
2. Clona el repositorio de Detectron2 (DensePose vive en `projects/DensePose` y sus configuraciones no se instalan con pip).
3. Compila Detectron2 con CUDA para la GPU del pod, con una barra de progreso. Tarda entre 3 y 10 minutos.
4. Instala DensePose.
5. Verifica que todo importe correctamente.

Si algo falla, la celda se detiene en ese paso y muestra las últimas líneas del error. El registro completo queda en `instalacion.log`. Si Detectron2 ya está instalado y funciona, se salta la compilación.

In [ ]:
import os, sys, re, time, shutil, subprocess, importlib

WORKDIR = "/workspace/detectron2_demo" if os.path.isdir("/workspace") else os.path.abspath("detectron2_demo")
REPO = os.path.join(os.path.dirname(WORKDIR), "detectron2_repo")
os.makedirs(WORKDIR, exist_ok=True)
PY = sys.executable                      # el Python del kernel: se instala donde el notebook importa
LOG = os.path.join(WORKDIR, "instalacion.log")
open(LOG, "w").close()


def ejecutar(cmd, titulo, env=None, progreso=False):
    print(f"\n[{titulo}]")
    entorno = os.environ.copy()
    entorno.update(env or {})
    inicio = time.time()
    ultimas, n_lineas, total = [], 0, 0
    with open(LOG, "a") as log:
        log.write(f"\n===== {titulo}\n$ {' '.join(cmd)}\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=entorno, bufsize=1)
        for linea in proc.stdout:
            log.write(linea)
            n_lineas += 1
            ultimas = (ultimas + [linea.rstrip()])[-40:]
            if not progreso:
                continue
            seg = int(time.time() - inicio)
            m = re.match(r"\s*\[(\d+)/(\d+)\]", linea)   # ninja imprime [archivo/total]
            if m:
                actual, total = int(m.group(1)), int(m.group(2))
                llenos = int(30 * actual / total)
                barra = "#" * llenos + "-" * (30 - llenos)
                print(f"\r  compilando [{barra}] {actual}/{total} archivos  {seg} s   ", end="", flush=True)
            elif total == 0 and n_lineas % 25 == 0:
                print(f"\r  preparando la compilación...  {seg} s   ", end="", flush=True)
        proc.wait()
    if progreso:
        print()
    if proc.returncode != 0:
        print(f"  ERROR (código {proc.returncode}). Últimas líneas:")
        print("\n".join("    " + l for l in ultimas))
        raise RuntimeError(f"Falló el paso '{titulo}'. Registro completo en {LOG}")
    print(f"  listo ({int(time.time() - inicio)} s)")


def importa(codigo):
    return subprocess.run([PY, "-c", codigo], capture_output=True, text=True).returncode == 0


# 0. Entorno
import torch
assert torch.cuda.is_available(), "No se detecta GPU: revisa que el pod tenga GPU asignada"
mayor, menor = torch.cuda.get_device_capability(0)
ARQ = f"{mayor}.{menor}"
CUDA_HOME = "/usr/local/cuda" if os.path.isdir("/usr/local/cuda") else os.environ.get("CUDA_HOME", "")
print("Python:", PY)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "| arquitectura:", ARQ)
print("CUDA_HOME:", CUDA_HOME or "(no encontrado)")

# 1. Dependencias
ejecutar([PY, "-m", "pip", "install", "-q", "--ignore-installed", "blinker"], "1/5 Arreglo de blinker")
ejecutar([PY, "-m", "pip", "install", "-q", "ninja", "opencv-python-headless", "pycocotools", "flask",
          "flask-cors", "requests", "matplotlib", "av", "scipy"], "1/5 Dependencias")

# 2. Repositorio
if not os.path.isdir(REPO):
    ejecutar(["git", "clone", "-q", "https://github.com/facebookresearch/detectron2.git", REPO],
             "2/5 Clonando Detectron2")
else:
    print("\n[2/5 Repositorio] ya existe, se reutiliza")

# 3. Compilación de Detectron2
if importa("from detectron2 import _C"):
    print("\n[3/5 Detectron2] ya está instalado y compilado, se omite")
else:
    shutil.rmtree(os.path.join(REPO, "build"), ignore_errors=True)   # restos de intentos anteriores
    env = {
        "FORCE_CUDA": "1",
        "TORCH_CUDA_ARCH_LIST": ARQ,            # compilar solo para esta GPU es mucho más rápido
        "MAX_JOBS": str(os.cpu_count() or 4),
        "PATH": f"{CUDA_HOME}/bin:" + os.environ.get("PATH", ""),
    }
    if CUDA_HOME:
        env["CUDA_HOME"] = CUDA_HOME
    ejecutar([PY, "-m", "pip", "install", "-v", "--no-build-isolation", REPO],
             "3/5 Compilando Detectron2", env=env, progreso=True)

# 4. DensePose (--no-deps evita que intente reinstalar detectron2 desde GitHub)
ejecutar([PY, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", "--force-reinstall",
          os.path.join(REPO, "projects", "DensePose")], "4/5 Instalando DensePose")

# 5. Verificación
print("\n[5/5 Verificación]")
importlib.invalidate_caches()
import detectron2
from detectron2 import _C
from densepose import add_densepose_config
print("  detectron2", detectron2.__version__, "->", os.path.dirname(detectron2.__file__))
print("  extensiones CUDA: OK")
print("  DensePose: OK")
print("\nInstalación completa. Continúa con la sección 2 (no hace falta reiniciar el kernel).")

## 2. Importaciones y utilidades

`crear_predictor` arma una configuración a partir del Model Zoo: carga el archivo `.yaml` de la arquitectura y la URL de los pesos preentrenados en COCO. `DefaultPredictor` se encarga de redimensionar la imagen, pasarla por el modelo y devolver las predicciones **en las coordenadas de la imagen original**.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import os, sys, time
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.data import MetadataCatalog, Metadata
from detectron2.utils.visualizer import Visualizer

WORKDIR = "/workspace/detectron2_demo" if os.path.isdir("/workspace") else os.path.abspath("detectron2_demo")
REPO = os.path.join(os.path.dirname(WORKDIR), "detectron2_repo")
os.chdir(WORKDIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
UMBRAL = 0.5
print("Dispositivo:", DEVICE)

MODELOS = {
    "deteccion":  "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml",
    "instancias": "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml",
    "panoptica":  "COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml",
    "keypoints":  "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml",
}

def crear_predictor(config_path, umbral=UMBRAL):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(config_path))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = umbral
    cfg.MODEL.DEVICE = DEVICE
    predictor = DefaultPredictor(cfg)
    metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])
    return predictor, metadata

def mostrar(img_rgb, titulo="", tam=(12, 8)):
    plt.figure(figsize=tam)
    plt.imshow(img_rgb)
    plt.title(titulo)
    plt.axis("off")
    plt.show()

## 3. Imagen de prueba

Se usa una imagen del conjunto de validación de COCO con personas y caballos. Para probar con una imagen propia, súbela al directorio de trabajo y cambia `RUTA_IMAGEN`.

Detectron2 trabaja con imágenes en formato **BGR** (convención de OpenCV); para mostrarlas con matplotlib se invierten los canales a RGB.

In [ ]:
RUTA_IMAGEN = "input.jpg"
if not os.path.exists(RUTA_IMAGEN):
    !wget -q http://images.cocodataset.org/val2017/000000439715.jpg -O {RUTA_IMAGEN}

img = cv2.imread(RUTA_IMAGEN)          # BGR, forma (alto, ancho, 3)
img_rgb = img[:, :, ::-1]
print("Tamaño:", img.shape)
mostrar(img_rgb, "Imagen original")

## 4. Detección de objetos (Faster R-CNN)

La salida es un diccionario con la clave `instances`. Cada instancia tiene:

- `pred_boxes`: caja en formato `[x1, y1, x2, y2]` (esquina superior izquierda e inferior derecha, en píxeles).
- `pred_classes`: índice de la clase en las 80 categorías de COCO.
- `scores`: confianza entre 0 y 1.

Es la misma información que devuelve YOLO.

In [ ]:
pred_det, meta_det = crear_predictor(MODELOS["deteccion"])

t0 = time.perf_counter()
salida = pred_det(img)
print(f"Inferencia: {(time.perf_counter() - t0) * 1000:.0f} ms")

inst = salida["instances"].to("cpu")
print("Objetos detectados:", len(inst))
for caja, clase, score in zip(inst.pred_boxes.tensor.numpy(), inst.pred_classes.numpy(), inst.scores.numpy()):
    nombre = meta_det.thing_classes[clase]
    print(f"  {nombre:<12} {score:.2f}  caja = {np.round(caja, 1).tolist()}")

v = Visualizer(img_rgb, meta_det, scale=1.0)
mostrar(v.draw_instance_predictions(inst).get_image(), "Detección de objetos")

## 5. Segmentación de instancias (Mask R-CNN)

Además de la caja, cada instancia trae `pred_masks`: una matriz booleana del tamaño de la imagen (`alto x ancho`) que vale `True` en los píxeles del objeto. Cada objeto tiene su propia máscara, por eso se distinguen individuos de la misma clase.

Una máscara no son coordenadas, son píxeles. Para enviarla por una API conviene convertirla en **polígonos** (contornos), que sí son listas de coordenadas `(x, y)`. Eso se hace con `cv2.findContours`.

In [ ]:
pred_ins, meta_ins = crear_predictor(MODELOS["instancias"])
salida = pred_ins(img)
inst = salida["instances"].to("cpu")

mascaras = inst.pred_masks.numpy()
print("Forma de pred_masks:", mascaras.shape, "(instancias, alto, ancho)")

def mascara_a_poligonos(mascara, epsilon=1.0, area_min=4):
    # RETR_CCOMP devuelve contornos exteriores y también los huecos interiores
    contornos, _ = cv2.findContours(mascara.astype(np.uint8), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    poligonos = []
    for c in contornos:
        if cv2.contourArea(c) < area_min:
            continue
        c = cv2.approxPolyDP(c, epsilon, True)
        if len(c) >= 3:
            poligonos.append(c.reshape(-1, 2).tolist())
    return poligonos

for i, (clase, score, m) in enumerate(zip(inst.pred_classes.numpy(), inst.scores.numpy(), mascaras)):
    polis = mascara_a_poligonos(m)
    n_puntos = sum(len(p) for p in polis)
    print(f"  {i + 1}. {meta_ins.thing_classes[clase]:<10} {score:.2f}  área = {int(m.sum())} px  "
          f"polígonos = {len(polis)}  puntos = {n_puntos}")

v = Visualizer(img_rgb, meta_ins, scale=1.0)
mostrar(v.draw_instance_predictions(inst).get_image(), "Segmentación de instancias")

## 6. Segmentación panóptica (Panoptic FPN)

La salida es `panoptic_seg`, una tupla con:

- un mapa `alto x ancho` donde cada píxel tiene el **id del segmento** al que pertenece (0 = sin asignar);
- `segments_info`: una lista que describe cada id: si es un objeto contable (`isthing=True`, por ejemplo una persona) o fondo (`isthing=False`, por ejemplo pasto o cielo), su categoría y su área.

Los objetos contables usan las 80 clases de COCO (`thing_classes`) y el fondo usa 54 clases de "material" (`stuff_classes`).

In [ ]:
pred_pan, meta_pan = crear_predictor(MODELOS["panoptica"])
salida = pred_pan(img)
panoptic_seg, segments_info = salida["panoptic_seg"]
panoptic_seg = panoptic_seg.to("cpu").numpy()

print("Forma del mapa panóptico:", panoptic_seg.shape)
print("Segmentos:", len(segments_info))
for s in segments_info:
    if s["isthing"]:
        nombre = meta_pan.thing_classes[s["category_id"]]
        tipo = "objeto"
    else:
        nombre = meta_pan.stuff_classes[s["category_id"]]
        tipo = "fondo "
    area = int((panoptic_seg == s["id"]).sum())   # los objetos no traen "area" en segments_info
    print(f"  id={s['id']:<3} {tipo}  {nombre:<14} área = {area}")

v = Visualizer(img_rgb, meta_pan, scale=1.0)
mostrar(v.draw_panoptic_seg(torch.from_numpy(panoptic_seg), segments_info).get_image(), "Segmentación panóptica")

## 7. Segmentación semántica

El Model Zoo de Detectron2 para COCO no incluye un modelo exclusivamente semántico, pero Panoptic FPN tiene internamente una rama semántica. La forma más clara de obtener una segmentación semántica completa es **colapsar la salida panóptica**: a cada píxel se le asigna la clase de su segmento y se descarta qué instancia era. Así, tres personas distintas pasan a ser una sola región "person".

Se arma un espacio de clases unificado: índices 0 a 79 para objetos y 80 en adelante para fondo. El valor 255 indica "sin clase".

In [ ]:
def panoptico_a_semantico(panoptic_seg, segments_info, meta):
    nombres = list(meta.thing_classes) + list(meta.stuff_classes)
    n_obj = len(meta.thing_classes)
    sem = np.full(panoptic_seg.shape, 255, dtype=np.int32)
    for s in segments_info:
        clase = s["category_id"] if s["isthing"] else n_obj + s["category_id"]
        sem[panoptic_seg == s["id"]] = clase
    return sem, nombres

sem, nombres_sem = panoptico_a_semantico(panoptic_seg, segments_info, meta_pan)
clases, areas = np.unique(sem[sem != 255], return_counts=True)
print("Clases presentes:")
for c, a in zip(clases, areas):
    print(f"  {nombres_sem[c]:<14} {a / sem.size * 100:5.1f} % de la imagen")

meta_sem = Metadata().set(stuff_classes=nombres_sem)
v = Visualizer(img_rgb, meta_sem, scale=1.0)
mostrar(v.draw_sem_seg(sem, alpha=0.6).get_image(), "Segmentación semántica")

## 8. Keypoints (Keypoint R-CNN)

Para cada persona detectada, `pred_keypoints` tiene forma `(personas, 17, 3)`: 17 puntos del cuerpo definidos por COCO, cada uno con `(x, y, confianza)`. Los nombres de los puntos y qué pares se unen para formar el "esqueleto" vienen en la metadata (`keypoint_names` y `keypoint_connection_rules`).

In [ ]:
pred_kp, meta_kp = crear_predictor(MODELOS["keypoints"])
salida = pred_kp(img)
inst = salida["instances"].to("cpu")

kps = inst.pred_keypoints.numpy()
print("Forma de pred_keypoints:", kps.shape)
print("Nombres:", meta_kp.keypoint_names)

if len(kps) > 0:
    print("Primera persona:")
    for nombre, (x, y, conf) in zip(meta_kp.keypoint_names, kps[0]):
        print(f"  {nombre:<16} x={x:7.1f}  y={y:7.1f}  conf={conf:.2f}")

v = Visualizer(img_rgb, meta_kp, scale=1.0)
mostrar(v.draw_instance_predictions(inst).get_image(), "Keypoints")

## 9. DensePose

DensePose va más allá de los keypoints: para **cada píxel** del cuerpo predice

- `I`: a cuál de las 24 partes del cuerpo pertenece (0 = fondo),
- `U, V`: coordenadas entre 0 y 1 sobre la superficie de esa parte en un modelo 3D del cuerpo humano (SMPL).

La salida cruda de la red se convierte con `DensePoseResultExtractor`, que devuelve, para cada persona, un mapa de etiquetas y un mapa UV del tamaño **de su caja**. Por eso, para ubicarlo en la imagen hace falta la caja `(x, y, ancho, alto)`.

In [ ]:
sys.path.insert(0, os.path.join(REPO, "projects", "DensePose"))
from densepose import add_densepose_config
from densepose.vis.extractor import DensePoseResultExtractor

def crear_predictor_densepose(umbral=UMBRAL):
    cfg = get_cfg()
    add_densepose_config(cfg)
    cfg.merge_from_file(os.path.join(REPO, "projects", "DensePose", "configs", "densepose_rcnn_R_50_FPN_s1x.yaml"))
    cfg.MODEL.WEIGHTS = ("https://dl.fbaipublicfiles.com/densepose/"
                         "densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl")
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = umbral
    cfg.MODEL.DEVICE = DEVICE
    return DefaultPredictor(cfg)

PARTES_DENSEPOSE = [
    "fondo", "torso posterior", "torso frontal", "mano derecha", "mano izquierda",
    "pie izquierdo", "pie derecho", "muslo der. posterior", "muslo izq. posterior",
    "muslo der. frontal", "muslo izq. frontal", "pierna der. posterior", "pierna izq. posterior",
    "pierna der. frontal", "pierna izq. frontal", "brazo izq. frontal", "brazo der. frontal",
    "brazo izq. posterior", "brazo der. posterior", "antebrazo izq. frontal", "antebrazo der. frontal",
    "antebrazo izq. posterior", "antebrazo der. posterior", "cabeza derecha", "cabeza izquierda",
]

pred_dp = crear_predictor_densepose()
salida = pred_dp(img)
inst = salida["instances"].to("cpu")
resultados_dp, cajas_xywh = DensePoseResultExtractor()(inst)
print("Personas:", len(inst))

In [ ]:
# Paleta fija de 25 colores (uno por parte) a partir del mapa de color JET
PALETA = cv2.applyColorMap((np.arange(25) * 10).astype(np.uint8).reshape(-1, 1), cv2.COLORMAP_JET).reshape(25, 3)

def pegar_en_caja(lienzo, parche, alfa_mask, x, y):
    # Pega 'parche' (h, w, 3) sobre 'lienzo' en (x, y) respetando los bordes de la imagen
    alto, ancho = lienzo.shape[:2]
    h, w = parche.shape[:2]
    x0, y0 = max(x, 0), max(y, 0)
    x1, y1 = min(x + w, ancho), min(y + h, alto)
    if x1 <= x0 or y1 <= y0:
        return
    p = parche[y0 - y:y1 - y, x0 - x:x1 - x]
    m = alfa_mask[y0 - y:y1 - y, x0 - x:x1 - x][..., None]
    zona = lienzo[y0:y1, x0:x1]
    lienzo[y0:y1, x0:x1] = np.where(m, (0.35 * zona + 0.65 * p).astype(np.uint8), zona)

vis_partes = img.copy()
vis_u = img.copy()
vis_v = img.copy()

for i, (res, caja) in enumerate(zip(resultados_dp, cajas_xywh)):
    etiquetas = res.labels.cpu().numpy()          # (h, w) con valores 0..24
    uv = res.uv.cpu().numpy()                     # (2, h, w) con valores 0..1
    x, y = int(caja[0]), int(caja[1])
    cuerpo = etiquetas > 0

    pegar_en_caja(vis_partes, PALETA[etiquetas], cuerpo, x, y)
    pegar_en_caja(vis_u, cv2.applyColorMap((uv[0] * 255).astype(np.uint8), cv2.COLORMAP_JET), cuerpo, x, y)
    pegar_en_caja(vis_v, cv2.applyColorMap((uv[1] * 255).astype(np.uint8), cv2.COLORMAP_JET), cuerpo, x, y)

    partes = [PARTES_DENSEPOSE[p] for p in np.unique(etiquetas) if p > 0]
    print(f"Persona {i + 1}: caja xywh = {np.round(caja.numpy(), 1).tolist()}  mapa = {etiquetas.shape}")
    print("   partes visibles:", ", ".join(partes))

fig, ejes = plt.subplots(1, 3, figsize=(18, 6))
for eje, im, titulo in zip(ejes, [vis_partes, vis_u, vis_v], ["Partes del cuerpo (I)", "Coordenada U", "Coordenada V"]):
    eje.imshow(im[:, :, ::-1])
    eje.set_title(titulo)
    eje.axis("off")
plt.show()

## 10. Resumen: ¿qué devuelve cada tarea?

| Tarea | Tipo de salida | Cómo se envía por la API |
|---|---|---|
| Detección | Coordenadas directas | Cajas `[x1, y1, x2, y2]` |
| Instancias | Máscara por objeto | Caja + polígonos (contornos de la máscara) |
| Semántica | Mapa de clases por píxel | Polígonos por clase |
| Panóptica | Mapa de segmentos por píxel | Polígonos por segmento |
| Keypoints | Coordenadas directas | 17 puntos `(x, y, confianza)` + conexiones del esqueleto |
| DensePose | Mapa denso dentro de cada caja | Caja + imagen PNG transparente con las partes coloreadas |

Los polígonos incluyen también los huecos de la máscara; en el navegador se rellenan con la regla `evenodd` para que los huecos queden vacíos.

Antes de levantar la API conviene liberar la memoria de la GPU que usaron las celdas anteriores, porque la API corre en un proceso aparte y carga sus propios modelos.

In [ ]:
for nombre in ["pred_det", "pred_ins", "pred_pan", "pred_kp", "pred_dp"]:
    if nombre in globals():
        del globals()[nombre]
import gc
gc.collect()
torch.cuda.empty_cache()
print("Memoria GPU liberada")

## 11. API Flask

La API tiene tres rutas:

- `GET /` sirve `index.html` si está en el mismo directorio que `app.py`.
- `GET /salud` indica si el servidor está activo y en qué dispositivo corre.
- `POST /detectar` recibe un formulario `multipart/form-data` con los campos `imagen` (archivo), `tipo` (`deteccion`, `instancias`, `semantica`, `panoptica`, `keypoints` o `densepose`) y opcionalmente `umbral` (0 a 1).

Los modelos se cargan la primera vez que se piden y quedan en memoria. La primera petición de cada tipo tarda más porque descarga los pesos. Un candado (`Lock`) evita que dos peticiones usen la GPU al mismo tiempo.

In [ ]:
%%writefile app.py
import os
import sys
import time
import base64
import threading

import numpy as np
import cv2
import torch
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.data import MetadataCatalog

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
REPO = os.environ.get("DETECTRON2_REPO", os.path.join(os.path.dirname(BASE_DIR), "detectron2_repo"))
PUERTO = int(os.environ.get("PUERTO", "8000"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
UMBRAL_BASE = 0.3      # los modelos se crean con este umbral y luego se filtra según la petición

TIPOS = ["deteccion", "instancias", "semantica", "panoptica", "keypoints", "densepose"]
CONFIGS = {
    "deteccion":  "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml",
    "instancias": "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml",
    "panoptica":  "COCO-PanopticSegmentation/panoptic_fpn_R_101_3x.yaml",
    "keypoints":  "COCO-Keypoints/keypoint_rcnn_R_50_FPN_3x.yaml",
}
PARTES_DENSEPOSE = [
    "fondo", "torso posterior", "torso frontal", "mano derecha", "mano izquierda",
    "pie izquierdo", "pie derecho", "muslo der. posterior", "muslo izq. posterior",
    "muslo der. frontal", "muslo izq. frontal", "pierna der. posterior", "pierna izq. posterior",
    "pierna der. frontal", "pierna izq. frontal", "brazo izq. frontal", "brazo der. frontal",
    "brazo izq. posterior", "brazo der. posterior", "antebrazo izq. frontal", "antebrazo der. frontal",
    "antebrazo izq. posterior", "antebrazo der. posterior", "cabeza derecha", "cabeza izquierda",
]
PALETA = cv2.applyColorMap((np.arange(25) * 10).astype(np.uint8).reshape(-1, 1), cv2.COLORMAP_JET).reshape(25, 3)

_modelos = {}
_lock = threading.Lock()


# ---------------------------------------------------------------- modelos
def _crear_zoo(config_path):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(config_path))
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_path)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = UMBRAL_BASE
    cfg.MODEL.DEVICE = DEVICE
    return DefaultPredictor(cfg), MetadataCatalog.get(cfg.DATASETS.TRAIN[0])


def _crear_densepose():
    sys.path.insert(0, os.path.join(REPO, "projects", "DensePose"))
    from densepose import add_densepose_config
    from densepose.vis.extractor import DensePoseResultExtractor
    cfg = get_cfg()
    add_densepose_config(cfg)
    cfg.merge_from_file(os.path.join(REPO, "projects", "DensePose", "configs", "densepose_rcnn_R_50_FPN_s1x.yaml"))
    cfg.MODEL.WEIGHTS = ("https://dl.fbaipublicfiles.com/densepose/"
                         "densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl")
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = UMBRAL_BASE
    cfg.MODEL.DEVICE = DEVICE
    return DefaultPredictor(cfg), DensePoseResultExtractor()


def _modelo(tipo):
    clave = "panoptica" if tipo == "semantica" else tipo
    if clave not in _modelos:
        print("Cargando modelo:", clave, flush=True)
        _modelos[clave] = _crear_densepose() if clave == "densepose" else _crear_zoo(CONFIGS[clave])
    return _modelos[clave]


# ---------------------------------------------------------------- utilidades
def _poligonos(mascara, epsilon=1.0, area_min=4):
    contornos, _ = cv2.findContours(mascara.astype(np.uint8), cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    salida = []
    for c in contornos:
        if cv2.contourArea(c) < area_min:
            continue
        c = cv2.approxPolyDP(c, epsilon, True)
        if len(c) >= 3:
            salida.append(c.reshape(-1, 2).tolist())
    return salida


def _caja(t):
    return [round(float(v), 1) for v in t]


def _png_base64(bgra):
    ok, buf = cv2.imencode(".png", bgra)
    return "data:image/png;base64," + base64.b64encode(buf.tobytes()).decode("ascii")


# ---------------------------------------------------------------- tareas
def tarea_deteccion(img, umbral, incluir_mascaras=False):
    predictor, meta = _modelo("instancias" if incluir_mascaras else "deteccion")
    inst = predictor(img)["instances"].to("cpu")
    resultados = []
    for i in range(len(inst)):
        score = float(inst.scores[i])
        if score < umbral:
            continue
        r = {
            "clase": meta.thing_classes[int(inst.pred_classes[i])],
            "score": round(score, 3),
            "caja": _caja(inst.pred_boxes.tensor[i]),
        }
        if incluir_mascaras:
            m = inst.pred_masks[i].numpy()
            r["area"] = int(m.sum())
            r["poligonos"] = _poligonos(m)
        resultados.append(r)
    return {"resultados": resultados}


def _panoptico(img, umbral):
    predictor, meta = _modelo("panoptica")
    seg, info = predictor(img)["panoptic_seg"]
    seg = seg.to("cpu").numpy()
    info = [s for s in info if not s["isthing"] or s.get("score", 1.0) >= umbral]
    return seg, info, meta


def tarea_panoptica(img, umbral):
    seg, info, meta = _panoptico(img, umbral)
    resultados = []
    for s in info:
        es_objeto = bool(s["isthing"])
        mascara = seg == s["id"]
        nombre = meta.thing_classes[s["category_id"]] if es_objeto else meta.stuff_classes[s["category_id"]]
        r = {
            "id": int(s["id"]),
            "clase": nombre,
            "es_objeto": es_objeto,
            "area": int(mascara.sum()),
            "poligonos": _poligonos(mascara),
        }
        if es_objeto and "score" in s:
            r["score"] = round(float(s["score"]), 3)
        resultados.append(r)
    return {"resultados": resultados}


def tarea_semantica(img, umbral):
    seg, info, meta = _panoptico(img, umbral)
    grupos = {}
    for s in info:
        nombre = meta.thing_classes[s["category_id"]] if s["isthing"] else meta.stuff_classes[s["category_id"]]
        grupos.setdefault(nombre, []).append(s["id"])
    resultados = []
    for nombre, ids in grupos.items():
        mascara = np.isin(seg, ids)
        resultados.append({"clase": nombre, "area": int(mascara.sum()), "poligonos": _poligonos(mascara)})
    resultados.sort(key=lambda r: -r["area"])
    return {"resultados": resultados}


def tarea_keypoints(img, umbral):
    predictor, meta = _modelo("keypoints")
    inst = predictor(img)["instances"].to("cpu")
    nombres = list(meta.keypoint_names)
    resultados = []
    for i in range(len(inst)):
        score = float(inst.scores[i])
        if score < umbral:
            continue
        puntos = []
        for nombre, (x, y, conf) in zip(nombres, inst.pred_keypoints[i].numpy()):
            puntos.append({"nombre": nombre, "x": round(float(x), 1), "y": round(float(y), 1),
                           "score": round(float(conf), 3)})
        resultados.append({"clase": "person", "score": round(score, 3),
                           "caja": _caja(inst.pred_boxes.tensor[i]), "puntos": puntos})
    conexiones = [[a, b] for a, b, _ in meta.keypoint_connection_rules]
    return {"resultados": resultados, "conexiones": conexiones}


def tarea_densepose(img, umbral):
    predictor, extractor = _modelo("densepose")
    inst = predictor(img)["instances"].to("cpu")
    if len(inst) == 0:
        return {"resultados": []}
    datos, cajas_xywh = extractor(inst)
    resultados = []
    for i, (res, caja) in enumerate(zip(datos, cajas_xywh)):
        score = float(inst.scores[i])
        if score < umbral:
            continue
        etiquetas = res.labels.cpu().numpy().astype(np.int64)
        alfa = np.where(etiquetas > 0, 255, 0).astype(np.uint8)
        bgra = np.dstack([PALETA[etiquetas], alfa])
        x, y, w, h = [float(v) for v in caja]
        resultados.append({
            "clase": "person",
            "score": round(score, 3),
            "caja": [round(x, 1), round(y, 1), round(x + w, 1), round(y + h, 1)],
            "mapa_tamano": [int(etiquetas.shape[1]), int(etiquetas.shape[0])],
            "imagen_png": _png_base64(bgra),
            "partes": [PARTES_DENSEPOSE[p] for p in np.unique(etiquetas) if p > 0],
        })
    return {"resultados": resultados}


TAREAS = {
    "deteccion": lambda img, u: tarea_deteccion(img, u, incluir_mascaras=False),
    "instancias": lambda img, u: tarea_deteccion(img, u, incluir_mascaras=True),
    "semantica": tarea_semantica,
    "panoptica": tarea_panoptica,
    "keypoints": tarea_keypoints,
    "densepose": tarea_densepose,
}


# ---------------------------------------------------------------- Flask
app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 20 * 1024 * 1024
CORS(app)


@app.get("/")
def inicio():
    if os.path.exists(os.path.join(BASE_DIR, "index.html")):
        return send_from_directory(BASE_DIR, "index.html")
    return jsonify({"mensaje": "API Detectron2 activa. Sube index.html junto a app.py para usar la interfaz."})


@app.get("/salud")
def salud():
    return jsonify({"ok": True, "dispositivo": DEVICE, "tipos": TIPOS, "cargados": list(_modelos.keys())})


@app.post("/detectar")
def detectar():
    tipo = request.form.get("tipo", "").strip().lower()
    if tipo not in TAREAS:
        return jsonify({"error": "tipo no válido", "tipos": TIPOS}), 400
    archivo = request.files.get("imagen")
    if archivo is None:
        return jsonify({"error": "falta el campo 'imagen'"}), 400
    try:
        umbral = float(request.form.get("umbral", "0.5"))
    except ValueError:
        return jsonify({"error": "umbral no válido"}), 400
    umbral = min(max(umbral, UMBRAL_BASE), 1.0)

    datos = np.frombuffer(archivo.read(), dtype=np.uint8)
    img = cv2.imdecode(datos, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({"error": "no se pudo leer la imagen"}), 400

    try:
        with _lock:
            t0 = time.perf_counter()
            respuesta = TAREAS[tipo](img, umbral)
            tiempo = (time.perf_counter() - t0) * 1000
    except Exception as ex:
        return jsonify({"error": str(ex)}), 500

    respuesta.update({"tipo": tipo, "ancho": int(img.shape[1]), "alto": int(img.shape[0]),
                      "umbral": umbral, "tiempo_ms": round(tiempo, 1)})
    return jsonify(respuesta)


if __name__ == "__main__":
    if os.environ.get("PRECARGAR") == "1":
        for t in ["deteccion", "instancias", "panoptica", "keypoints", "densepose"]:
            _modelo(t)
    app.run(host="0.0.0.0", port=PUERTO, threaded=True)

### Levantar la API en segundo plano

La API corre como un proceso independiente (`nohup`), así el notebook queda libre. Los mensajes del servidor se guardan en `api.log`. Con `PRECARGAR=1` se cargan los cinco modelos al arrancar, para que la primera petición no tarde.

Para usar la interfaz web, sube `index.html` a este mismo directorio (arrastrándolo al explorador de archivos de Jupyter) antes o después de levantar la API.

In [ ]:
import subprocess, requests

!pkill -f "python app.py" || true
proceso = subprocess.Popen("PRECARGAR=1 nohup python app.py > api.log 2>&1 &", shell=True, cwd=WORKDIR)

URL_LOCAL = "http://localhost:8000"
print("Esperando a que la API cargue los modelos (1 a 3 minutos)...")
lista = False
for intento in range(150):
    try:
        lista = requests.get(URL_LOCAL + "/salud", timeout=2).ok
    except requests.exceptions.RequestException:
        lista = False
    if lista:
        break
    if proceso.poll() is not None and intento > 5 and "Traceback" in open("api.log").read():
        break
    time.sleep(2)

if not lista:
    print("La API no respondió. Últimas líneas de api.log:")
    !tail -n 30 api.log
    raise RuntimeError("La API no arrancó; revisa el error de arriba")
print(requests.get(URL_LOCAL + "/salud").json())

pod_id = os.environ.get("RUNPOD_POD_ID")
if pod_id:
    print("URL pública de la API:", f"https://{pod_id}-8000.proxy.runpod.net")
else:
    print("No se encontró RUNPOD_POD_ID; la API está en", URL_LOCAL)

### Probar la API desde Python

Se envía la imagen de prueba con cada tipo de tarea y se muestra cuántos resultados devolvió y un fragmento del JSON.

In [ ]:
import json

for tipo in ["deteccion", "instancias", "semantica", "panoptica", "keypoints", "densepose"]:
    with open(RUTA_IMAGEN, "rb") as f:
        r = requests.post(URL_LOCAL + "/detectar", files={"imagen": f},
                          data={"tipo": tipo, "umbral": "0.5"}, timeout=600)
    d = r.json()
    if not r.ok:
        print(tipo, "ERROR", d)
        continue
    print(f"{tipo:<11} resultados = {len(d['resultados']):<3} tiempo = {d['tiempo_ms']} ms")

# Ejemplo de la respuesta de keypoints (primera persona, primeros tres puntos)
with open(RUTA_IMAGEN, "rb") as f:
    d = requests.post(URL_LOCAL + "/detectar", files={"imagen": f}, data={"tipo": "keypoints"}).json()
if d["resultados"]:
    ejemplo = dict(d["resultados"][0])
    ejemplo["puntos"] = ejemplo["puntos"][:3]
    print(json.dumps(ejemplo, indent=2, ensure_ascii=False))

### Ver el registro y detener la API

In [ ]:
!tail -n 20 api.log

In [ ]:
# Cambia a True y ejecuta esta celda solo cuando quieras detener la API.
# Queda en False para que "Run All" no apague la API al final.
DETENER_API = False
if DETENER_API:
    !pkill -f "python app.py" || true
    print("API detenida")
else:
    print("La API sigue corriendo. Cambia DETENER_API a True para detenerla.")